# "But wait... there's more"

## A More Visible Agent Loop

The Digital Twin contained an Agent Loop. But it was behind-the-scenes, running every time the user asked a message. Using its tools and then replying. It didn't feel very... loopy.

### Adding 2 more ingredients to make it more real

Let's make an Agent Loop with some familiar features borrowed from Claude Code:

1. A Terminal UI (TUI)
2. A Checklist tool to cause and track multiple tool calls


In [1]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv(override=True)

True

In [ ]:
def Show(Text):
    try:
        Console().print(Text)
    except Exception:
        print (Text)

Show("My name is Siddharth Begwani")

My name is Siddharth Begwani

In [3]:
openAI = OpenAI()

In [4]:
# Some lists!
Checklist = []
Completed = []

In [5]:
def GetChecklistReport() -> str:
    Result = ""
    for Index, Item in enumerate(Checklist):
        if Completed[Index]:
            Result += f"Checklist #{Index+1}: [green][strike]{Item}[/strike][/green]\n"
        else:
            Result += f"Checklist #{Index+1}: {Item}\n"
    Show (Result)
    return Result

In [6]:
GetChecklistReport()

''

In [7]:
def CreateChecklist(Descriptions: list[str]) -> str:
    Checklist.extend(Descriptions)
    Completed.extend([False] * len(Descriptions))
    return GetChecklistReport()

In [8]:
def MarkComplete(Index:int, CompletionNotes: str) -> str:
    if 1 <= Index <= len(Checklist):
        Completed[Index-1] = True
    else:
        return "No checklist at this index."
    Console().print(CompletionNotes)
    return GetChecklistReport()

In [9]:
Checklist, Completed = [], []
CreateChecklist(["Buy icecream", "Finish passport application", "Continue intermittent fasting"])

Checklist #1: Buy icecream
Checklist #2: Finish passport application
Checklist #3: Continue intermittent fasting

'Checklist #1: Buy icecream\nChecklist #2: Finish passport application\nChecklist #3: Continue intermittent fasting\n'

In [10]:
MarkComplete(1,"Bought")

Bought

Checklist #1: Buy icecream
Checklist #2: Finish passport application
Checklist #3: Continue intermittent fasting

'Checklist #1: [green][strike]Buy icecream[/strike][/green]\nChecklist #2: Finish passport application\nChecklist #3: Continue intermittent fasting\n'

In [12]:
CreateChecklistJSON = {
    "name": "CreateChecklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "Descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
            }
        },
        "required": ["Descriptions"],
        "additionalProperties": False
    }
}

MarkCompleteJSON = {
    "name": "MarkComplete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        "properties": {
            'Index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
            },
            'CompletionNotes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
            }
        },
        "required": ['Index','CompletionNotes'],
        "type": "object",
        "additionalProperties": False
    }
}

In [13]:
Tools = [{"type":"function", "function": CreateChecklistJSON},
         {"type":"function", "function": MarkCompleteJSON}]

In [18]:
def HandleToolCalls(ToolCalls):
    Results = []
    for ToolCall in ToolCalls:
        ToolName = ToolCall.function.name
        Arguments = json.loads(ToolCall.function.arguments)
        Tool = globals().get(ToolName)
        Result = Tool(**Arguments) if Tool else {}
        Results.append({"role":"tool", "content": json.dumps(Result), "tool_call_id":ToolCall.id})
    return Results

In [19]:
SystemPrompt = """
                You are given a problem to solve, by using your checklist tools to plan 
                a list of steps, then carrying out each step in turn.
                Now create a plan, set the checklist, carry out the steps and then reply
                with the solution.
                If any quantity is not provided in the problem specified, then include a step
                to come up with a reasonable estimate.
                Provide your solution in Rich console markup without code blocks.
                Do not ask user questions or clarification; respond only with the answer after
                using your tools.
                """
UserPrompt = """
             A train leaves Boston at 2:00 p.m. travelling at 60 mph.
             Another train leaves New York at 3:00 p.m. travelling 80 mph towards Boston.
             When do they meet?
             """

Messages = [{"role":"system", "content":SystemPrompt}, {"role":"user", "content":UserPrompt}]

In [20]:
def Loop(Messages):
    Response = openAI.chat.completions.create(model="gpt-5.5", messages=Messages, tools=Tools)
    while Response.choices[0].finish_reason == "tool_calls":
        Message = Response.choices[0].message
        Messages.append(Message)
        Results = HandleToolCalls(Message.tool_calls)
        Messages.extend(Results)
        Response = openAI.chat.completions.create(model="gpt-5.5", messages=Messages, tools=Tools)
    Show(Response.choices[0].message.content)

In [21]:
Checklist, Completed = [], []
Loop (Messages)

Checklist #1: Identify known quantities: departure times, speeds, and route/distance needed.
Checklist #2: Estimate or choose a reasonable Boston–New York rail distance because it is not explicitly provided.
Checklist #3: Compute Boston train's head start distance by 3:00 p.m.
Checklist #4: Compute remaining distance between trains at 3:00 p.m. and their combined closing speed.
Checklist #5: Calculate time after 3:00 p.m. until they meet and convert to clock time.
Checklist #6: Prepare final answer with the estimate clearly stated.

Known: Boston train leaves at 2:00 p.m. at 60 mph. New York train leaves at 3:00 p.m. at 80 mph toward Boston. Need
the Boston–New York distance, which is not specified.

Checklist #1: Identify known quantities: departure times, speeds, and route/distance needed.
Checklist #2: Estimate or choose a reasonable Boston–New York rail distance because it is not explicitly provided.
Checklist #3: Compute Boston train's head start distance by 3:00 p.m.
Checklist #4: Compute remaining distance between trains at 3:00 p.m. and their combined closing speed.
Checklist #5: Calculate time after 3:00 p.m. until they meet and convert to clock time.
Checklist #6: Prepare final answer with the estimate clearly stated.

The problem omits the distance, so use a reasonable estimate: Boston to New York is about 215 miles by rail/road 
corridor.

Checklist #1: Identify known quantities: departure times, speeds, and route/distance needed.
Checklist #2: Estimate or choose a reasonable Boston–New York rail distance because it is not explicitly provided.
Checklist #3: Compute Boston train's head start distance by 3:00 p.m.
Checklist #4: Compute remaining distance between trains at 3:00 p.m. and their combined closing speed.
Checklist #5: Calculate time after 3:00 p.m. until they meet and convert to clock time.
Checklist #6: Prepare final answer with the estimate clearly stated.

...

Time after 3:00 p.m. is 155 ÷ 140 = 1.107 hours, which is about 1 hour 6 minutes. So they meet at about 4:06 p.m..

Checklist #1: Identify known quantities: departure times, speeds, and route/distance needed.
Checklist #2: Estimate or choose a reasonable Boston–New York rail distance because it is not explicitly provided.
Checklist #3: Compute Boston train's head start distance by 3:00 p.m.
Checklist #4: Compute remaining distance between trains at 3:00 p.m. and their combined closing speed.
Checklist #5: Calculate time after 3:00 p.m. until they meet and convert to clock time.
Checklist #6: Prepare final answer with the estimate clearly stated.

Final response will state that because the distance is omitted, the answer depends on the assumed Boston–New York 
distance; using 215 miles, meeting time is about 4:06 p.m..

Checklist #1: Identify known quantities: departure times, speeds, and route/distance needed.
Checklist #2: Estimate or choose a reasonable Boston–New York rail distance because it is not explicitly provided.
Checklist #3: Compute Boston train's head start distance by 3:00 p.m.
Checklist #4: Compute remaining distance between trains at 3:00 p.m. and their combined closing speed.
Checklist #5: Calculate time after 3:00 p.m. until they meet and convert to clock time.
Checklist #6: Prepare final answer with the estimate clearly stated.

The exact meeting time cannot be determined unless the Boston–New York distance is given.

Using a reasonable estimate of 215 miles between Boston and New York:

• Boston train gets a 1-hour head start:  
  60 mph × 1 hour = 60 miles

• Remaining distance at 3:00 p.m.:  
  215 − 60 = 155 miles

• Combined speed after 3:00 p.m.:  
  60 + 80 = 140 mph

• Time to meet after 3:00 p.m.:  
  155 ÷ 140 ≈ 1.11 hours ≈ 1 hour 6 minutes

They meet at approximately 4:06 p.m.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>